# 10 — Tratamento: coleta → limpeza → refinamento

Aqui o dado vira dataset. Duas etapas, com papeis distintos:

| Camada | Pergunta que responde | Operacoes tipicas |
|---|---|---|
| `limpeza` | *o dado esta correto?* | tipos, nulos, duplicatas, padronizacao, validacao |
| `refinamento` | *o dado esta pronto para uso?* | joins, agregacoes, regras de negocio, colunas derivadas |

O Streamlit le **so do refinamento**. Manter as camadas separadas permite corrigir uma regra de
negocio sem reprocessar a ingestao, e auditar em que ponto um numero mudou.

In [ ]:
from pyspark.sql import functions as F
from lakehouse import sessao, gravar, perfil, listar

spark = sessao("10-tratamento")
listar(spark, "coleta")

## Etapa 1 — limpeza

Carregue da coleta e aplique as correcoes. Os blocos abaixo sao um catalogo: use o que se aplica.

In [ ]:
bruto = spark.table("nessie.coleta.clientes")
perfil(bruto)

In [ ]:
limpo = bruto

# --- tipos ---------------------------------------------------------------
# limpo = limpo.withColumn("valor", F.col("valor").cast("double"))
# limpo = limpo.withColumn("data",  F.to_date("data", "dd/MM/yyyy"))

# --- texto ---------------------------------------------------------------
limpo = limpo.withColumn("email", F.lower(F.trim(F.col("email"))))
# remove acento e uniformiza caixa (util para chave de join)
# limpo = limpo.withColumn("nome_chave",
#             F.upper(F.translate(F.trim("customer_name"),
#                                 "áàâãéêíóôõúçÁÀÂÃÉÊÍÓÔÕÚÇ",
#                                 "aaaaeeioooucAAAAEEIOOOUC")))

# --- nulos ---------------------------------------------------------------
# limpo = limpo.fillna({"uf": "NAO INFORMADO", "valor": 0.0})
# limpo = limpo.dropna(subset=["customer_id"])       # sem chave, a linha nao serve

# --- duplicatas ----------------------------------------------------------
# Mantem a linha mais recente por chave. dropDuplicates sozinho nao garante QUAL fica.
# from pyspark.sql.window import Window
# janela = Window.partitionBy("customer_id").orderBy(F.col("created_at").desc())
# limpo = (limpo.withColumn("_rn", F.row_number().over(janela))
#               .filter("_rn = 1").drop("_rn"))
limpo = limpo.dropDuplicates(["customer_id"])

# --- auditoria -----------------------------------------------------------
limpo = limpo.withColumn("_carregado_em", F.current_timestamp())

print(f"  {bruto.count()} linhas na coleta  ->  {limpo.count()} apos limpeza")

### Validar antes de gravar

Grave lixo na limpeza e ele chega no dashboard. Uma checagem simples evita isso:

In [ ]:
regras = {
    "sem id nulo":        limpo.filter(F.col("customer_id").isNull()).count() == 0,
    "id unico":           limpo.select("customer_id").distinct().count() == limpo.count(),
    "email com @":        limpo.filter(~F.col("email").contains("@")).count() == 0,
    "tem linhas":         limpo.count() > 0,
}
for regra, passou in regras.items():
    print(f"  {'OK  ' if passou else 'FALHOU'} {regra}")

if not all(regras.values()):
    raise ValueError("validacao falhou — corrija antes de gravar")

In [ ]:
gravar(limpo, "limpeza.clientes", modo="substituir")

## Etapa 2 — refinamento

Junte as fontes ja limpas e produza a tabela que o dashboard consome. Aqui entram as regras de
negocio: faixas, classificacoes, metricas.

In [ ]:
clientes = spark.table("nessie.limpeza.clientes")

# Exemplo: juntar com outra tabela limpa
# pedidos = spark.table("nessie.limpeza.pedidos")
# base = clientes.join(pedidos, clientes.email == pedidos.cliente_email, "left")

refinado = (clientes
    .withColumn("dominio_email", F.split(F.col("email"), "@").getItem(1))
    .withColumn("faixa_cadastro",
        F.when(F.col("created_at") >= F.date_sub(F.current_date(), 30),  "novo")
         .when(F.col("created_at") >= F.date_sub(F.current_date(), 365), "recente")
         .otherwise("antigo"))
)

refinado.show(5, truncate=40)

### Agregacao para o dashboard

Agregue **aqui**, nao no Streamlit. O dashboard deve ler poucas linhas ja prontas — assim ele
responde rapido e a regra fica versionada no lakehouse, nao espalhada pelo codigo da tela.

In [ ]:
resumo = (refinado
    .groupBy("dominio_email", "faixa_cadastro")
    .agg(F.count("*").alias("qtd_clientes"))
    .orderBy(F.col("qtd_clientes").desc()))

resumo.show(truncate=False)

In [ ]:
gravar(refinado, "refinamento.clientes", modo="substituir")
gravar(resumo,   "refinamento.clientes_por_dominio", modo="substituir")

## Particionar tabelas grandes

Acima de alguns milhoes de linhas, particione por uma coluna usada em filtro (data, quase sempre).
O Spark e o Dremio passam a ler so as particoes necessarias:

```python
gravar(vendas, "refinamento.vendas", particao="ano_mes")
```

Nao particione por coluna de alta cardinalidade (id, e-mail): gera milhares de arquivinhos e piora
a leitura em vez de melhorar.

In [ ]:
listar(spark)

Pronto para o dashboard. Confira o resultado em **`20_publicar.ipynb`**.